In [ ]:
# ============================================================
# MCT-LTDiag  —  v5  STAGE 1: LIVER
# DS²Net + MHA Self-Attention (Swin-Tiny backbone)
#
# Architecture
#   Encoder  : Swin-Tiny  (20-ch pseudo-3D, 5 ctx slices × 4 phases)
#   Bottleneck: SEM@d4 → MHA Self-Attention (14×14, 8 heads) → conv4
#   Decoder  : DEM@d3 → DEM@d2 → DEM@d1 → dec0
#   Heads    : 5 × Conv2d(256→1)  — DS² deep supervision
#   Output   : (B, 1, H, W)  liver probability map
#   Loss     : DS² adaptive  wIoU + wBCE  (+boundary @ head0 only)
#
# Stage 1 outputs written per-case for Stage 2:
#   {LIVER_DIR}/{case_id}_liver_prob.npy   float32 (H, W, D)
#   {LIVER_DIR}/{case_id}_liver_mask.npy   uint8   (H, W, D)
#
# Fixes applied (vs previous version)
# FIX-1  _weighted_iou_bce: kernel hard-capped at 31; weight map
#         clamped [1,5] — eliminates NaN/Inf overflow in bf16
# FIX-2  remap_to_npy: checks NPY_CACHE directly so delete_npz=True
#         never causes silent Drive-npz fallback during training
# FIX-3  WeightedRandomSampler keyed on per-slice liver pixel area
#         saved in Cell 7; prevents edge-slice under-sampling
# FIX-4  copy_cache_to_tmp_parallel: completeness check validates
#         both file count AND per-file byte sizes (catches truncation)
# FIX-5  scheduler.state_dict() saved + restored on resume so LR
#         curve is continuous across interruptions
# FIX-6  Gradient checkpointing via timm unified interface with
#         attribute-level fallback for older timm versions
# FIX-7  Cell 5 Drive scan: pvp.exists()/mask.exists() wrapped in
#         OSError retry for errno 103/107 (FUSE connection drop)
# FIX-8  connected_component_filter: removed unreachable
#         n_components==0 branch (already guarded by sum==0 above)
# FIX-9  val_loader shuffle=True — prevents all slices of one case
#         arriving in the same batch and skewing BN statistics
# FIX-10 _preprocess_one_liver: np.clip(vol, 0, 1) after normalisation
#         and after sk_resize — prevents spline overshoot (order=1) from
#         producing extreme float32 values (e.g. 7e32) that failed the
#         data sanity check even though finite=True
# FIX-11 check_data_clean: removed the imgs.max()<=1.1 range gate;
#         now only checks NaN/Inf (the genuine failure mode) and warns
#         separately for values above 2.0 (true normalisation failure)
# FIX-12 boundary_loss: replaced incorrect conv-based boundary detection
#         with correct morphological dilation−erosion via max_pool2d;
#         old code found interior points not boundary pixels
# FIX-13 UD flip: separated from LR flip with aug_ud_flip_prob=0.05;
#         abdominal CT is cranio-caudally oriented so UD flip at 0.5
#         creates anatomically impossible training samples
# FIX-14 DS2NetLiver.forward: added assert that Swin backbone returns
#         NHWC so any future timm version change fails loudly not silently
# FIX-15 Scheduler resume: last_epoch vs total_steps guard prevents
#         IndexError when resuming late in training with few epochs left
# ============================================================


# %% ── CELL 1: Install ────────────────────────────────────────────────────────
!pip install -q nibabel timm scipy scikit-learn tqdm scikit-image


# %% ── CELL 2: Imports ────────────────────────────────────────────────────────
import os, random, time, hashlib, math as _math
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from scipy.ndimage import zoom, gaussian_filter, map_coordinates, label as cc_label
from scipy.ndimage import rotate as scipy_rotate
from skimage.transform import resize as sk_resize
from tqdm.notebook import tqdm
import timm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    torch.backends.cudnn.benchmark        = True
    torch.set_float32_matmul_precision("high")
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
else:
    print("WARNING: No GPU detected.")


# %% ── CELL 3: Mount Drive & paths ───────────────────────────────────────────
from google.colab import drive

def ensure_drive_mounted(max_retries=3):
    for attempt in range(1, max_retries + 1):
        if not os.path.isdir('/content/drive/MyDrive'):
            print(f"  Mounting Drive (attempt {attempt}) ...")
            try:
                drive.mount('/content/drive')
            except (ValueError, OSError) as e:
                print(f"  Initial mount failed (attempt {attempt}): {e}")
        try:
            _ = os.listdir('/content/drive/MyDrive')
            print(f"  Drive mounted (attempt {attempt}) ✓")
            return True
        except OSError as e:
            print(f"  Drive FUSE error (attempt {attempt}): {e}")
        # Always try force_remount on any failure
        try:
            drive.mount('/content/drive', force_remount=True)
            _ = os.listdir('/content/drive/MyDrive')
            print(f"  Drive remounted ✓")
            return True
        except (ValueError, OSError) as e:
            # ValueError: mount failed — DriveFS daemon crashed
            # OSError: FUSE transport error
            print(f"  Remount failed (attempt {attempt}): {e}")
        if attempt < max_retries:
            print(f"  Waiting 10s before retry ..."); time.sleep(10)
    raise RuntimeError("Google Drive could not be mounted after retries.")

ensure_drive_mounted()

SAVE_DIR  = Path("/content/drive/MyDrive/MCT_LTDiag/v_download")
CKPT_DIR  = SAVE_DIR / "seg_pvt_checkpoints_liver"
CACHE_DIR = SAVE_DIR / "seg_cache_20ch_224"
LIVER_DIR = CKPT_DIR / "liver_masks"      # Stage 2 reads from here

assert SAVE_DIR.exists(), f"SAVE_DIR not found: {SAVE_DIR}"
for d in [CKPT_DIR, CACHE_DIR, LIVER_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"SAVE_DIR  : {SAVE_DIR}  ✓")
print(f"CKPT_DIR  : {CKPT_DIR}  ✓")
print(f"CACHE_DIR : {CACHE_DIR}  ✓")
print(f"LIVER_DIR : {LIVER_DIR}  ✓")


# %% ── CELL 4: Config ─────────────────────────────────────────────────────────
CFG = {
    # Model
    "backbone"            : "swin_tiny_patch4_window7_224",
    "backbone_channels"   : (96, 192, 384, 768),
    "n_classes"           : 1,          # liver only
    "mha_heads"           : 8,
    "mha_dropout"         : 0.1,
    "img_size"            : 224,
    # Data
    "n_context_slices"    : 5,          # OPT-B: 5 ctx slices × 4 phases = 20ch
    "n_phases"            : 4,
    "hu_windows" : {
        "nc.nii.gz"    : (-100, 200),
        "art.nii.gz"   : (-100, 300),
        "pvp.nii.gz"   : (-100, 250),
        "delay.nii.gz" : (-100, 200),
    },
    "target_spacing"      : 1.0,
    "train_ratio"         : 0.70,
    "seed"                : 42,
    # Training
    "batch_size"          : 48,
    "grad_accum_steps"    : 4,          # effective batch = 192
    "lr"                  : 1e-4,
    "epochs"              : 60,
    "early_stop_patience" : 15,
    "pct_start"           : 0.20,
    "pos_weight"          : 5.0,
    "boundary_weight"     : 0.3,
    "max_grad_norm"       : 1.0,
    "val_every"           : 3,
    "preprocess_workers"  : 24,
    # Augmentation (nnU-Net style)
    "aug_noise_std"       : 0.03,
    "aug_noise_prob"      : 0.15,
    "aug_blur_sigma"      : (0.5, 1.5),
    "aug_blur_prob"       : 0.20,
    "aug_gamma_range"     : (0.7, 1.5),
    "aug_gamma_prob"      : 0.30,
    "aug_elastic_alpha"   : 100,
    "aug_elastic_sigma"   : 10,
    "aug_elastic_prob"    : 0.20,
    "aug_scale_range"     : (0.75, 1.25),
    "aug_scale_prob"      : 0.20,
    "aug_mirror_prob"     : 0.50,
    "aug_rotation_max_deg": 30,
    "aug_rotation_prob"   : 0.30,
    "aug_ud_flip_prob"    : 0.05,   # FIX-13: separate from LR flip (abdominal CT is cranio-caudally oriented)
    # Inference
    "seg_threshold"       : 0.5,
    "min_component_voxels": {"default": 1000},
}
CFG["n_input_channels"] = CFG["n_phases"] * CFG["n_context_slices"]  # 20

random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])

METRIC_KEYS     = ["Dice", "IoU", "Precision", "Recall", "F1", "MAE"]
PHASE_FILES_SEG = ["nc.nii.gz", "art.nii.gz", "pvp.nii.gz", "delay.nii.gz"]

print(f"Backbone       : {CFG['backbone']}")
print(f"Input channels : {CFG['n_input_channels']}  "
      f"({CFG['n_phases']} phases x {CFG['n_context_slices']} ctx)")
print(f"Batch size     : {CFG['batch_size']} x accum {CFG['grad_accum_steps']} "
      f"= {CFG['batch_size']*CFG['grad_accum_steps']} effective")


# %% ── CELL 5: Load metadata ──────────────────────────────────────────────────
# FIX-7: .exists() calls on Drive paths wrapped in OSError retry so that
# errno 103 (Software caused connection abort) and errno 107 (Transport
# endpoint not connected) don't silently skip valid cases.

tab_path = SAVE_DIR / "meta_info_patient.tab"
for _ in range(30):
    if tab_path.exists(): break
    time.sleep(1)
else:
    _ = list(SAVE_DIR.iterdir()); time.sleep(3)
    assert tab_path.exists(), "meta_info_patient.tab not found"

patient_df = pd.read_csv(tab_path, sep="\t")
print(f"Patient metadata: {len(patient_df)} rows")

records = []
for case_dir in sorted(SAVE_DIR.iterdir()):
    if not case_dir.is_dir():
        continue
    pvp  = case_dir / "NIFTI" / "pvp.nii.gz"
    mask = case_dir / "mask_pvp.nii.gz"
    try:                                         # FIX-7
        has_pvp, has_mask = pvp.exists(), mask.exists()
    except OSError as e:
        if e.errno in (103, 107):
            print(f"  Drive dropped at {case_dir.name} — remounting ...")
            ensure_drive_mounted()
            try:
                has_pvp, has_mask = pvp.exists(), mask.exists()
            except Exception:
                continue
        else:
            raise
    if not has_pvp or not has_mask:
        continue
    row = patient_df[patient_df["ID"].astype(str) == case_dir.name]
    if row.empty:
        continue
    records.append({
        "case_id"    : case_dir.name,
        "pvp_path"   : str(pvp),
        "mask_path"  : str(mask),
        "tumor_type" : str(row.iloc[0]["type"]).strip().upper(),
    })

df = pd.DataFrame(records)
print(f"Total cases with pvp + mask: {len(df)}")
print(df["tumor_type"].value_counts())


# %% ── CELL 6: Balanced stratified split ─────────────────────────────────────
def balanced_three_way_split(df, train_ratio, seed):
    rng = np.random.RandomState(seed)
    train_rows, val_rows, test_rows = [], [], []
    for _, group in df.groupby("tumor_type"):
        idx = group.index.tolist(); rng.shuffle(idx)
        n             = len(idx)
        n_valtest     = n - int(round(n * train_ratio))
        n_val         = n_valtest // 2
        n_test        = n_valtest // 2
        n_train_final = n - n_val - n_test
        train_rows.extend(idx[:n_train_final])
        val_rows.extend(idx[n_train_final : n_train_final + n_val])
        test_rows.extend(idx[n_train_final + n_val :
                              n_train_final + n_val + n_test])
    return (df.loc[train_rows].reset_index(drop=True),
            df.loc[val_rows].reset_index(drop=True),
            df.loc[test_rows].reset_index(drop=True))

train_df, val_df, test_df = balanced_three_way_split(df, CFG["train_ratio"], CFG["seed"])
print(f"Split — train:{len(train_df)} | val:{len(val_df)} | test:{len(test_df)}")
comp = pd.DataFrame({
    "train": train_df["tumor_type"].value_counts(),
    "val"  : val_df["tumor_type"].value_counts(),
    "test" : test_df["tumor_type"].value_counts(),
}).fillna(0).astype(int)
print(comp)
assert (comp["val"] == comp["test"]).all(), "val/test counts not balanced"
print("val == test per class ✓")


# %% ── CELL 7: Preprocessing — 20-channel liver cache ────────────────────────
# Outputs per case (npz):
#   imgs  (D_liver, 20, H, W)  float32  — only slices containing liver
#   masks (D_liver,  1, H, W)  float32  — binary liver mask (label >= 1)
#   areas (D_liver,)           int32    — liver pixel count per slice
#                                         FIX-3: used by WeightedRandomSampler
#
# Hash key "liver_stage1_v2" avoids collision with any old joint-model cache.

def _preprocess_one_liver(args):
    pvp_path, mask_path, cache_dir, cfg = args
    n_ctx   = cfg["n_context_slices"]
    hu_wins = cfg["hu_windows"]

    key     = hashlib.md5(
        (pvp_path + mask_path + str(n_ctx) + "liver_stage1_v2").encode()
    ).hexdigest()[:12]
    cache_p = Path(cache_dir) / f"{key}.npz"
    if cache_p.exists():
        return str(cache_p), "cached"

    try:
        case_dir   = Path(pvp_path).parent.parent
        seg        = nib.load(mask_path).get_fdata(dtype=np.float32).astype(np.uint8)
        pvp_img    = nib.load(pvp_path)
        pvp_vol    = pvp_img.get_fdata(dtype=np.float32)
        spacing    = tuple(float(z) for z in pvp_img.header.get_zooms()[:3])
        factors    = tuple(s / cfg["target_spacing"] for s in spacing)

        phase_vols = []
        for fname in PHASE_FILES_SEG:
            hu_min, hu_max = hu_wins[fname]
            ph = case_dir / "NIFTI" / fname
            vol = nib.load(str(ph)).get_fdata(dtype=np.float32) if ph.exists() else pvp_vol.copy()
            if vol.shape != pvp_vol.shape:
                vol = zoom(vol, tuple(p/v for p,v in zip(pvp_vol.shape, vol.shape)), order=1)
            if any(abs(f-1.0) > 0.05 for f in factors):
                vol = zoom(vol, factors, order=1)
            vol = (np.clip(vol, hu_min, hu_max) - hu_min) / (hu_max - hu_min + 1e-8)
            # Hard clamp to [0,1]: zoom/interpolation can produce values slightly
            # outside the window boundary (spline overshoot), which when stored as
            # float32 and later read back can appear as extreme values like 7e32.
            vol = np.clip(vol, 0.0, 1.0).astype(np.float32)
            phase_vols.append(vol)

        if any(abs(f-1.0) > 0.05 for f in factors):
            seg = zoom(seg, factors, order=0).astype(np.uint8)
        if seg.max() == 0:
            return None, "empty"

        D, sz, half = phase_vols[0].shape[2], (cfg["img_size"],)*2, n_ctx // 2
        imgs_out, masks_out, areas_out = [], [], []

        for z in range(D):
            liver = (seg[:, :, z] >= 1).astype(np.float32)
            if liver.max() == 0:
                continue
            channels = []
            for dz in range(-half, half + 1):
                zc = max(0, min(D-1, z+dz))
                for vol in phase_vols:
                    sl = sk_resize(vol[:,:,zc], sz, order=1,
                                   preserve_range=True).astype(np.float32)
                    # sk_resize bilinear can overshoot [0,1] at slice edges
                    channels.append(np.clip(sl, 0.0, 1.0))
            msk = sk_resize(liver, sz, order=0, preserve_range=True).astype(np.float32)
            imgs_out.append(np.stack(channels, axis=0))
            masks_out.append(msk[np.newaxis])
            areas_out.append(int(msk.sum()))

        if imgs_out:
            np.savez_compressed(cache_p,
                                imgs=np.stack(imgs_out),
                                masks=np.stack(masks_out),
                                areas=np.array(areas_out, dtype=np.int32))
        return str(cache_p), "new"
    except Exception as e:
        return None, f"ERROR: {e}"


def build_cache_liver(split_df, cfg, desc="Preprocessing"):
    args = [(row["pvp_path"], row["mask_path"], str(CACHE_DIR), cfg)
            for _, row in split_df.iterrows()]
    paths, n_new, n_cached, n_err = [], 0, 0, 0
    with ProcessPoolExecutor(max_workers=cfg["preprocess_workers"]) as ex:
        futs = {ex.submit(_preprocess_one_liver, a): a for a in args}
        for fut in tqdm(as_completed(futs), total=len(args), desc=desc):
            p, st = fut.result()
            if st and str(st).startswith("ERROR"):
                print(f"  Warning: {st}"); n_err += 1; continue
            if p and Path(p).exists():
                paths.append(p)
                if st == "cached": n_cached += 1
                else:              n_new    += 1
    print(f"  {desc}: {n_new} new | {n_cached} cached | {n_err} errors | {len(paths)} total")
    return paths


print(f"Cache: {len(list(CACHE_DIR.glob('*.npz')))} existing files")
train_cache = build_cache_liver(train_df, CFG, "Train cache (liver)")
val_cache   = build_cache_liver(val_df,   CFG, "Val cache (liver)")
test_cache  = build_cache_liver(test_df,  CFG, "Test cache (liver)")


# %% ── CELL 8: Augmentation ───────────────────────────────────────────────────

def elastic_transform_2d(img, mask, alpha, sigma):
    _, h, w = img.shape
    dx = gaussian_filter(np.random.randn(h, w), sigma) * alpha
    dy = gaussian_filter(np.random.randn(h, w), sigma) * alpha
    gx, gy = np.meshgrid(np.arange(w), np.arange(h))
    coords  = [np.clip(gy+dy, 0, h-1), np.clip(gx+dx, 0, w-1)]
    img_d  = np.stack([map_coordinates(img[c],  coords, order=1).astype(np.float32)
                       for c in range(img.shape[0])])
    mask_d = np.stack([map_coordinates(mask[c], coords, order=0).astype(np.float32)
                       for c in range(mask.shape[0])])
    return img_d, mask_d


def augment_nnunet_ct(img_t, mask_t, cfg):
    img = img_t.numpy().copy()
    mk  = mask_t.numpy()
    _, H, W = img.shape
    n_ph, n_ctx = cfg["n_phases"], cfg["n_context_slices"]

    if random.random() < cfg["aug_elastic_prob"]:
        img, mk = elastic_transform_2d(img, mk,
                                       cfg["aug_elastic_alpha"],
                                       cfg["aug_elastic_sigma"])
    if random.random() < cfg["aug_scale_prob"]:
        sc = random.uniform(*cfg["aug_scale_range"])
        nh, nw = int(H*sc), int(W*sc)
        ir = np.stack([sk_resize(img[c], (nh,nw), order=1,
                                 preserve_range=True).astype(np.float32)
                       for c in range(img.shape[0])])
        mr = np.stack([sk_resize(mk[c],  (nh,nw), order=0,
                                 preserve_range=True).astype(np.float32)
                       for c in range(mk.shape[0])])
        if sc > 1.0:
            r0,c0 = (nh-H)//2, (nw-W)//2
            img, mk = ir[:, r0:r0+H, c0:c0+W], mr[:, r0:r0+H, c0:c0+W]
        else:
            ph,pw = (H-nh)//2, (W-nw)//2
            img = np.pad(ir, ((0,0),(ph,H-nh-ph),(pw,W-nw-pw)), constant_values=0)
            mk  = np.pad(mr, ((0,0),(ph,H-nh-ph),(pw,W-nw-pw)), constant_values=0)
    if random.random() < cfg["aug_rotation_prob"]:
        ang = random.uniform(-cfg["aug_rotation_max_deg"], cfg["aug_rotation_max_deg"])
        img = np.stack([scipy_rotate(img[c], ang, axes=(0,1), reshape=False,
                                     order=1, mode='constant', cval=0.)
                        for c in range(img.shape[0])]).astype(np.float32)
        mk  = np.stack([scipy_rotate(mk[c],  ang, axes=(0,1), reshape=False,
                                     order=0, mode='constant', cval=0.)
                        for c in range(mk.shape[0])]).astype(np.float32)
    # LR flip: anatomically valid for axial CT (left-right symmetric)
    if random.random() < cfg["aug_mirror_prob"]:
        img = img[:,:,::-1].copy(); mk = mk[:,:,::-1].copy()
    # FIX-13  UD flip uses a separate low probability (default 0.05).
    # Abdominal CT has cranial-caudal orientation: the liver cross-section
    # near the diaphragm looks completely different from lower slices.
    # Using aug_mirror_prob=0.5 for UD introduces anatomically impossible
    # samples that hurt boundary precision. Keep at 0.05 or disable entirely.
    if random.random() < cfg.get("aug_ud_flip_prob", 0.05):
        img = img[:,::-1,:].copy(); mk = mk[:,::-1,:].copy()

    for ph in range(n_ph):
        ch = [ph + ctx*n_ph for ctx in range(n_ctx)]
        if random.random() < 0.15:
            s, sh = random.uniform(0.9,1.1), random.uniform(-0.05,0.05)
            for c in ch: img[c] = np.clip(img[c]*s + sh, 0, 1)
        if random.random() < cfg["aug_noise_prob"]:
            noise = np.random.randn(*img[0].shape).astype(np.float32) * cfg["aug_noise_std"]
            for c in ch: img[c] = np.clip(img[c]+noise, 0, 1)
        if random.random() < cfg["aug_blur_prob"]:
            sig = random.uniform(*cfg["aug_blur_sigma"])
            for c in ch:
                img[c] = np.clip(gaussian_filter(img[c], sig).astype(np.float32), 0, 1)
        if random.random() < cfg["aug_gamma_prob"]:
            g = random.uniform(*cfg["aug_gamma_range"])
            for c in ch:
                img[c] = np.clip(np.clip(img[c],1e-7,1.).astype(np.float64)**g, 0,1).astype(np.float32)
    # Final clamp: cache built before FIX-10 may contain spline-overshoot
    # values (e.g. 7e32). Clamp here as last line of defence before loss.
    img = np.clip(img, 0.0, 1.0)
    return torch.from_numpy(img).float(), torch.from_numpy(mk).float()


# %% ── CELL 9: Cache → /tmp, npz→npy, Dataset & DataLoaders ──────────────────
import shutil

TMP_CACHE = Path("/tmp/seg_cache_liver_224")
NPY_CACHE = Path("/tmp/seg_npy_liver_224")
NPY_CACHE.mkdir(exist_ok=True)


def _drive_alive():
    try: os.listdir('/content/drive/MyDrive'); return True
    except OSError: return False


def copy_cache_to_tmp_parallel(src_dir, dst_dir, n_workers=16, force=False):
    """
    Copy .npz files from Drive → /tmp.

    FIX-4: Completeness check validates per-file byte sizes (not just count)
    so files truncated during a previous disk-full event are caught and
    re-copied rather than silently treated as valid.
    """
    src_dir, dst_dir = Path(src_dir), Path(dst_dir)
    if not _drive_alive():
        print("  Drive disconnected — remounting ..."); ensure_drive_mounted()

    src_files = list(src_dir.glob("*.npz"))
    if not src_files:
        raise RuntimeError(f"No .npz files in {src_dir}")

    if dst_dir.exists() and not force:
        dst_files = list(dst_dir.glob("*.npz"))
        src_sz    = {f.name: f.stat().st_size for f in src_files}
        dst_sz    = {f.name: f.stat().st_size for f in dst_files}
        # FIX-4: require both count and sizes to match
        if (len(dst_files) == len(src_files) and
                all(dst_sz.get(n, 0) == s for n, s in src_sz.items())):
            print(f"  /tmp cache complete and verified ({len(dst_files)} files) ✓")
            return dst_dir
        bad = sum(1 for n,s in src_sz.items() if dst_sz.get(n,0) != s)
        print(f"  Incomplete/truncated ({len(dst_files)}/{len(src_files)} files, "
              f"{bad} size mismatches) — recopying ...")
        shutil.rmtree(dst_dir)

    dst_dir.mkdir(parents=True, exist_ok=True)
    total = sum(f.stat().st_size for f in src_files)
    print(f"  Copying {len(src_files)} files ({total/1e9:.1f} GB) ...")

    def _copy_one(src):
        try:
            shutil.copy(src, dst_dir / src.name)
            return src.stat().st_size
        except OSError as e:
            if e.errno in (103, 107):
                return f"ERR:{src.name}"
            raise

    t0, copied, errs = time.time(), 0, []
    for f in tqdm(src_files, desc="  Stat prefetch"):
        try: f.stat()
        except OSError: pass
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        futs = {ex.submit(_copy_one, f): f for f in src_files}
        for fut in tqdm(as_completed(futs), total=len(src_files), desc="  Copying"):
            r = fut.result()
            if isinstance(r, str) and r.startswith("ERR:"): errs.append(r[4:])
            else: copied += r
    print(f"  Done: {copied/1e9:.1f} GB in {(time.time()-t0)/60:.1f} min")
    if errs:
        ensure_drive_mounted()
        for fname in tqdm(errs, desc="  Re-copying"):
            try: shutil.copy(src_dir / fname, dst_dir / fname)
            except Exception as e: print(f"  Still failed: {fname}: {e}")
    return dst_dir


def convert_one_npz(npz_path, delete_npz=True):
    """
    Convert one /tmp .npz → NPY_CACHE (_imgs.npy, _masks.npy, _areas.npy).

    delete_npz=True frees the /tmp npz immediately after success so peak
    /tmp usage stays at ~1× cache size instead of ~2× (avoiding disk-full).
    The areas array is saved for the WeightedRandomSampler (FIX-3).
    """
    npz_path = Path(npz_path)
    stem     = npz_path.stem
    imgs_p, masks_p, areas_p = (NPY_CACHE / f"{stem}_{k}.npy"
                                 for k in ("imgs","masks","areas"))
    if imgs_p.exists() and masks_p.exists():
        if delete_npz: npz_path.unlink(missing_ok=True)
        return "cached"
    try:
        d = np.load(str(npz_path), allow_pickle=False)
        np.save(imgs_p,  d["imgs"])
        np.save(masks_p, d["masks"])
        if "areas" in d: np.save(areas_p, d["areas"])
        if delete_npz: npz_path.unlink(missing_ok=True)
        return "converted"
    except OSError as e:
        if e.errno in (103,107): return f"transport:{stem}"
    except Exception: pass
    # Recovery: re-copy from Drive and retry once
    drive_src = CACHE_DIR / npz_path.name
    try:
        if drive_src.exists():
            shutil.copy(drive_src, npz_path)
            d = np.load(str(npz_path), allow_pickle=False)
            np.save(imgs_p, d["imgs"]); np.save(masks_p, d["masks"])
            if "areas" in d: np.save(areas_p, d["areas"])
            if delete_npz: npz_path.unlink(missing_ok=True)
            return "recovered"
    except OSError as e:
        if e.errno in (103,107): return f"transport:{stem}"
    except Exception: pass
    for p in (npz_path, drive_src):
        try: p.unlink()
        except: pass
    return f"corrupt:{stem}"


TMP_CACHE = copy_cache_to_tmp_parallel(CACHE_DIR, TMP_CACHE)
src_files = list(TMP_CACHE.glob("*.npz"))
print(f"Converting {len(src_files)} npz → npy ...")
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as ex:
    results = list(tqdm(ex.map(convert_one_npz, src_files),
                        total=len(src_files), desc="Converting"))

cnt = {k: sum(1 for r in results if str(r).startswith(k))
       for k in ("converted","cached","recovered","transport","corrupt")}
print(f"  converted:{cnt['converted']} cached:{cnt['cached']} "
      f"recovered:{cnt['recovered']} transport-err:{cnt['transport']} "
      f"corrupt:{cnt['corrupt']}  ({time.time()-t0:.0f}s)")

transport_stems = [r.split(":",1)[1] for r in results if str(r).startswith("transport")]
if transport_stems:
    print(f"  Retrying {len(transport_stems)} transport errors ...")
    ensure_drive_mounted()
    for stem in tqdm(transport_stems, desc="  Retrying"):
        npz_p = TMP_CACHE / f"{stem}.npz"
        if not npz_p.exists():
            try: shutil.copy(CACHE_DIR / f"{stem}.npz", npz_p)
            except Exception as e: print(f"  Re-copy failed {stem}: {e}"); continue
        r = convert_one_npz(npz_p)
        if r not in ("converted","recovered","cached"):
            print(f"  Still failed {stem}: {r}")

corrupt_stems = {r.split(":",1)[1] for r in results if str(r).startswith("corrupt")}
if corrupt_stems:
    print(f"  WARNING: {len(corrupt_stems)} corrupt files excluded from all splits.")

n_ready = len(list(NPY_CACHE.glob("*_imgs.npy")))
print(f"  NPY cache: {n_ready} cases ready in {NPY_CACHE}")
if n_ready == 0:
    raise RuntimeError(
        "NPY cache is empty — disk was likely full during conversion.\n"
        "Fix:  !rm -rf /tmp/seg_cache_liver_224 /tmp/seg_npy_liver_224\n"
        "Then re-run this cell.")


def remap_to_npy(cache_list, npy_dir, skip_stems=None):
    """
    FIX-2: Return each case's original Drive path as the key string so
    LiverSegDataset can extract the stem and look up pre-converted .npy
    files in NPY_CACHE.

    The old implementation checked tmp_p.exists() which always returned
    False after delete_npz=True deleted the /tmp npz, silently falling
    back to slow Drive reads and triggering FUSE errors during training.
    Now we check NPY_CACHE directly and warn (but still include) cases
    where npy is missing — they will use the Drive npz fallback path in
    the Dataset.
    """
    out = []
    for p in cache_list:
        stem = Path(p).stem
        if skip_stems and stem in skip_stems:
            continue
        if not (Path(npy_dir)/f"{stem}_imgs.npy").exists():
            print(f"  Warning: npy missing for {stem} — Drive npz fallback")
        out.append(str(p))          # Drive path; Dataset resolves npy via stem
    return out


train_cache_rdy = remap_to_npy(train_cache, NPY_CACHE, corrupt_stems)
val_cache_rdy   = remap_to_npy(val_cache,   NPY_CACHE, corrupt_stems)
test_cache_rdy  = remap_to_npy(test_cache,  NPY_CACHE, corrupt_stems)


class LiverSegDataset(Dataset):
    """
    Lazy-loading dataset backed by memory-mapped .npy files.

    Index entry: (path_type, imgs_path, masks_path, slice_idx,
                  img_shape, img_dtype, mask_shape, mask_dtype)

    slice_weights: per-slice liver pixel area used by WeightedRandomSampler
    (FIX-3) so edge slices with small liver aren't under-sampled.
    """
    def __init__(self, cache_paths, augment=False, img_size=224, cfg=None):
        self.augment = augment; self.cfg = cfg
        self.index   = []; self.slice_weights = []
        n_missing_npy = 0

        for cp in tqdm(cache_paths, desc="  Indexing", leave=False):
            stem   = Path(cp).stem
            imgs_p = NPY_CACHE / f"{stem}_imgs.npy"
            msk_p  = NPY_CACHE / f"{stem}_masks.npy"
            area_p = NPY_CACHE / f"{stem}_areas.npy"

            if imgs_p.exists() and msk_p.exists():
                try:
                    mm  = np.load(str(imgs_p),  mmap_mode='r')
                    mm2 = np.load(str(msk_p),   mmap_mode='r')
                    n   = mm.shape[0]
                    areas = (np.load(str(area_p)) if area_p.exists()
                             else np.full(n, 1000, dtype=np.int32))
                    for i in range(n):
                        self.index.append(("npy", str(imgs_p), str(msk_p), i,
                                           mm.shape, mm.dtype, mm2.shape, mm2.dtype))
                        self.slice_weights.append(max(float(areas[i]), 100.))
                    del mm, mm2
                except Exception as e:
                    print(f"  Skipping corrupt npy {stem}: {e}")
                    for bad in [imgs_p, msk_p, area_p]: bad.unlink(missing_ok=True)
            else:
                n_missing_npy += 1
                try:
                    d = np.load(str(cp), allow_pickle=False)
                    n = d["imgs"].shape[0]
                    areas = (d["areas"] if "areas" in d
                             else np.full(n, 1000, dtype=np.int32))
                    for i in range(n):
                        self.index.append(("npz", str(cp), str(cp), i,
                                           d["imgs"].shape, d["imgs"].dtype,
                                           d["masks"].shape, d["masks"].dtype))
                        self.slice_weights.append(max(float(areas[i]), 100.))
                except Exception as e:
                    print(f"  Skipping corrupt npz {stem}: {e}")

        if n_missing_npy:
            print(f"  Warning: {n_missing_npy} case(s) using slow Drive npz fallback")
        split = 'train' if augment else 'val/test'
        print(f"  {split}: {len(self.index)} slices from {len(cache_paths)} cases")

    def __len__(self): return len(self.index)

    def __getitem__(self, idx):
        pt, ip, mp, si, ish, idt, msh, mdt = self.index[idx]
        if pt == "npy":
            mm  = np.memmap(ip, dtype=idt, mode='r', shape=ish)
            mm2 = np.memmap(mp, dtype=mdt, mode='r', shape=msh)
            img  = torch.from_numpy(mm[si].copy()).float().clamp(0, 1)
            mask = torch.from_numpy(mm2[si].copy()).float()
            del mm, mm2
        else:
            d = np.load(ip, allow_pickle=False)
            img  = torch.from_numpy(d["imgs"][si].copy()).float().clamp(0, 1)
            mask = torch.from_numpy(d["masks"][si].copy()).float()
        if self.augment and self.cfg:
            img, mask = augment_nnunet_ct(img, mask, self.cfg)
        return img, mask


def check_data_clean(loader, n_batches=20, name="train"):
    """
    Verify no NaN/Inf in the first n_batches.

    NOTE: we do NOT check imgs.max() <= 1.1 here.  sk_resize (bilinear,
    order=1) can produce values slightly outside [0,1] due to spline
    overshoot at slice boundaries.  The preprocessing pipeline now clips
    to [0,1] explicitly, but existing Drive cache files built before that
    fix may still contain marginal overshoots.  Since finite=True and the
    values are bounded float32 (not garbage), training is unaffected —
    the loss functions all work on logits, not raw pixel values.
    Only NaN/Inf is a genuine problem and is checked below.
    """
    print(f"Data sanity check — {name} ({n_batches} batches) ...")
    n_bad = 0
    for i, (imgs, masks) in enumerate(loader):
        if i >= n_batches: break
        imgs_ok  = torch.isfinite(imgs).all().item()
        masks_ok = torch.isfinite(masks).all().item()
        if not (imgs_ok and masks_ok):
            print(f"  Batch {i}: NaN/Inf detected  "
                  f"imgs_finite={imgs_ok}  masks_finite={masks_ok}")
            n_bad += 1
        elif imgs.max().item() > 1.01:
            # Values above 1.01 after clamping in __getitem__ means
            # the clamp did not apply — this should not happen.
            # Previously this was just a WARNING but 7e32 values caused
            # training to blow up (loss=2.87e26) so treat as hard error.
            print(f"  Batch {i}: CLAMP FAILURE  imgs.max()={imgs.max():.3f}")
            n_bad += 1
    if n_bad == 0:
        print(f"  {name} ✓ (all {n_batches} batches clean)")
    else:
        raise RuntimeError(
            f"{n_bad} batches with NaN/Inf in {name} loader.\n"
            "Fix:  !rm -rf /tmp/seg_npy_liver_224  then re-run Cell 9.")


train_ds = LiverSegDataset(train_cache_rdy, augment=True,  img_size=CFG["img_size"], cfg=CFG)
val_ds   = LiverSegDataset(val_cache_rdy,   augment=False, img_size=CFG["img_size"], cfg=CFG)
test_ds  = LiverSegDataset(test_cache_rdy,  augment=False, img_size=CFG["img_size"], cfg=CFG)
for ds, nm in [(train_ds,"train"),(val_ds,"val"),(test_ds,"test")]:
    if len(ds) == 0: raise RuntimeError(f"{nm} dataset is empty — re-run from Cell 1.")

# FIX-3: weight sampler by per-slice liver pixel area
sampler = WeightedRandomSampler(
    weights=torch.tensor(train_ds.slice_weights, dtype=torch.float),
    num_samples=len(train_ds.slice_weights), replacement=True)

train_loader = DataLoader(train_ds, CFG["batch_size"], sampler=sampler,
                          num_workers=8, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)
# FIX-9: shuffle=True prevents all slices of one case landing in the same batch
val_loader   = DataLoader(val_ds, CFG["batch_size"], shuffle=True,
                          num_workers=4, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)
test_loader  = DataLoader(test_ds, CFG["batch_size"], shuffle=False,
                          num_workers=4, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)

print(f"\nSlices — train:{len(train_ds)} | val:{len(val_ds)} | test:{len(test_ds)}")
check_data_clean(train_loader, n_batches=20, name="train")
check_data_clean(val_loader,   n_batches=10, name="val")


# %% ── CELL 10: Model — DS²Net + MHA ─────────────────────────────────────────

class MHASelfAttention(nn.Module):
    """
    Multi-Head Self-Attention at d4 (14×14 = 196 tokens), Pre-LN formulation.

    Pre-LN is more numerically stable than Post-LN at high LRs during early
    training.  ~790 K params (~2.5% of full model).

    Forward:
      x → flatten → LN → MHA(Q=K=V) → residual → LN → FFN → residual → reshape
    """
    def __init__(self, embed_dim=256, n_heads=8, dropout=0.1, ffn_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.mha   = nn.MultiheadAttention(embed_dim, n_heads, dropout=dropout,
                                           batch_first=True)
        self.drop1 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn   = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * ffn_ratio), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * ffn_ratio, embed_dim), nn.Dropout(dropout))

    def forward(self, x):
        B, D, H, W = x.shape
        t = x.flatten(2).permute(0,2,1)            # (B, HW, D)
        n = self.norm1(t)
        a, _ = self.mha(n, n, n)
        t = t + self.drop1(a)
        t = t + self.ffn(self.norm2(t))
        return t.permute(0,2,1).reshape(B, D, H, W)


class ConvBnRelu(nn.Module):
    def __init__(self, ic, oc, k=3, p=1):
        super().__init__()
        self.b = nn.Sequential(nn.Conv2d(ic,oc,k,padding=p,bias=False),
                               nn.BatchNorm2d(oc), nn.ReLU(inplace=True))
    def forward(self, x): return self.b(x)


class DEM(nn.Module):
    """
    Detail Enhancement Module (DS²Net §III-A, Fig. 4a).

    Detail mask M_d:
      channel_max(f_l) → Conv_k → Conv_k → Sigmoid

    Output:
      M_d ⊙ Conv3(Conv1(cat(up(f_h), f_l))) + residual(f_l)
    """
    def __init__(self, c_lo, c_hi, c_out, k=7):
        super().__init__()
        self.dc1 = nn.Sequential(nn.Conv2d(1, c_out, k, padding=k//2, bias=False),
                                  nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))
        self.dc2 = nn.Sequential(nn.Conv2d(c_out, c_out, k, padding=k//2, bias=False),
                                  nn.BatchNorm2d(c_out))
        self.sig = nn.Sigmoid()
        self.c1  = nn.Sequential(nn.Conv2d(c_hi+c_lo, c_out, 1, bias=False),
                                  nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))
        self.c3  = nn.Sequential(nn.Conv2d(c_out, c_out, 3, padding=1, bias=False),
                                  nn.BatchNorm2d(c_out))
        self.res = nn.Conv2d(c_lo, c_out, 1, bias=False) if c_lo != c_out else nn.Identity()

    def forward(self, f_l, f_h):
        Md    = self.sig(self.dc2(self.dc1(f_l.max(1,keepdim=True).values)))
        f_h_u = F.interpolate(f_h, f_l.shape[2:], mode='bilinear', align_corners=True)
        return Md * self.c3(self.c1(torch.cat([f_h_u, f_l], 1))) + self.res(f_l)


class SEM(nn.Module):
    """
    Semantic Enhancement Module (DS²Net §III-B, Fig. 4b).

    Spatial attention mask M_s:
      SA(up(f_h))  where SA = Conv7(cat(mean, max)) → Sigmoid

    Output:
      CA(M_s ⊙ proj(f_skip)) + proj_h(up(f_h))
    """
    def __init__(self, c_skip, c_prev, c_out):
        super().__init__()
        self.sa_p = nn.Conv2d(c_prev, c_out, 1, bias=False)
        self.sa_c = nn.Sequential(nn.Conv2d(2, 1, 7, padding=3, bias=False), nn.Sigmoid())
        self.pl   = nn.Conv2d(c_skip, c_out, 1, bias=False) if c_skip != c_out else nn.Identity()
        mid = max(c_out//16, 4)
        self.ca   = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                   nn.Linear(c_out, mid), nn.ReLU(inplace=True),
                                   nn.Linear(mid, c_out), nn.Sigmoid())
        self.ph   = nn.Conv2d(c_prev, c_out, 1, bias=False) if c_prev != c_out else nn.Identity()

    def forward(self, f_skip, f_prev):
        fu    = F.interpolate(f_prev, f_skip.shape[2:], mode='bilinear', align_corners=True)
        fp    = self.sa_p(fu)
        Ms    = self.sa_c(torch.cat([fp.mean(1,keepdim=True), fp.max(1,keepdim=True).values], 1))
        g     = Ms * self.pl(f_skip)
        caw   = self.ca(g).view(g.shape[0],-1,1,1)
        return g * caw + self.ph(fu)


class DS2NetLiver(nn.Module):
    """
    Pure DS²Net + MHA — liver-only single-channel output.

    Encoder    : Swin-Tiny  patch_embed expanded to n_input_channels
    Bottleneck : SEM@d4 → MHA self-attention → conv4 refinement
    Decoder    : DEM@d3 → DEM@d2 → DEM@d1 → dec0 (2× upsample)
    Heads      : 5 × Conv(D→1) for DS² deep supervision
    """
    def __init__(self, n_classes=1, ch=(96,192,384,768),
                 n_input_channels=20, D=256, mha_heads=8, mha_dropout=0.1):
        super().__init__()
        # ── Backbone ────────────────────────────────────────────────────────
        self.backbone = timm.create_model(
            'swin_tiny_patch4_window7_224', pretrained=True, features_only=True)
        old = self.backbone.patch_embed.proj
        new = nn.Conv2d(n_input_channels, old.out_channels,
                        kernel_size=old.kernel_size, stride=old.stride,
                        padding=old.padding, bias=old.bias is not None)
        with torch.no_grad():
            new.weight[:, :3] = old.weight
            new.weight[:, 3:] = old.weight.mean(1, keepdim=True).expand(
                -1, n_input_channels-3, -1, -1)
            if old.bias is not None: new.bias.copy_(old.bias)
        self.backbone.patch_embed.proj = new

        # FIX-6: timm unified gradient-checkpointing interface
        if hasattr(self.backbone, 'set_grad_checkpointing'):
            self.backbone.set_grad_checkpointing(enable=True)
        else:
            for layer in getattr(self.backbone, 'layers', []):
                for attr in ('gradient_checkpointing', 'use_checkpoint'):
                    if hasattr(layer, attr): setattr(layer, attr, True)

        # Store stage-0 channel count for FIX-14 assert in forward()
        self._stage0_ch = ch[0]

        # ── Bottleneck ──────────────────────────────────────────────────────
        self.mha_sa = MHASelfAttention(D, mha_heads, mha_dropout, ffn_ratio=4)

        # ── Decoder ─────────────────────────────────────────────────────────
        self.sem4  = SEM(ch[2], ch[3], D)
        self.conv4 = nn.Sequential(ConvBnRelu(D,D), ConvBnRelu(D,D))
        self.dem3  = DEM(ch[1], D, D, k=5)
        self.conv3 = nn.Sequential(ConvBnRelu(D,D), ConvBnRelu(D,D))
        self.dem2  = DEM(ch[0], D, D, k=7)
        self.conv2 = nn.Sequential(ConvBnRelu(D,D), ConvBnRelu(D,D))
        self.dem1  = DEM(ch[0], D, D, k=3)
        self.conv1 = nn.Sequential(ConvBnRelu(D,D), ConvBnRelu(D,D))
        self.dec0  = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            ConvBnRelu(D,D), ConvBnRelu(D,D))
        for i in range(5):
            setattr(self, f"head{i}", nn.Conv2d(D, n_classes, 1))

    def forward(self, x):
        H, W      = x.shape[2], x.shape[3]
        raw_feats = self.backbone(x)
        # FIX-14  Assert Swin outputs NHWC (last dim = channel).
        # timm features_only returns NHWC for Swin; if a future timm version
        # returns NCHW this assert fires immediately rather than silently
        # producing garbage from the wrong permute.
        # FIX-14 (strengthened): check last dim == known stage-0 channel count (96).
        # More robust than shape[-1]!=shape[1] which fails when H==C.
        # CFG["backbone_channels"][0] is always defined and needs no timm API.
        assert raw_feats[0].shape[-1] == self._stage0_ch, (
            f"Expected Swin NHWC output with last-dim={self._stage0_ch}, "
            f"got {raw_feats[0].shape} — check timm version (current assumes NHWC)")
        feats  = [f.permute(0,3,1,2).contiguous() for f in raw_feats]
        f1, f2, f3, f4 = feats

        d4 = self.conv4(self.mha_sa(self.sem4(f3, f4)))
        d3 = self.conv3(self.dem3(f2, F.interpolate(d4, scale_factor=2, mode='bilinear', align_corners=True)))
        d2 = self.conv2(self.dem2(f1, F.interpolate(d3, scale_factor=2, mode='bilinear', align_corners=True)))
        d1 = self.conv1(self.dem1(
            F.interpolate(f1, scale_factor=2, mode='bilinear', align_corners=True),
            F.interpolate(d2, scale_factor=2, mode='bilinear', align_corners=True)))
        d0 = self.dec0(d1)

        up = lambda f: F.interpolate(f, (H,W), mode='bilinear', align_corners=True)
        return [up(self.head4(d4)), up(self.head3(d3)),
                up(self.head2(d2)), up(self.head1(d1)), self.head0(d0)]


model = DS2NetLiver(
    n_classes=CFG["n_classes"], ch=CFG["backbone_channels"],
    n_input_channels=CFG["n_input_channels"],
    D=256, mha_heads=CFG["mha_heads"], mha_dropout=CFG["mha_dropout"]
).to(DEVICE)

n_tot  = sum(p.numel() for p in model.parameters())/1e6
n_bb   = sum(p.numel() for p in model.backbone.parameters())/1e6
n_mha  = sum(p.numel() for p in model.mha_sa.parameters())/1e3
bb_ids = {id(p) for p in model.backbone.parameters()}
n_dec  = sum(p.numel() for p in model.parameters() if id(p) not in bb_ids)/1e6
print(f"DS2Net-Liver  |  total:{n_tot:.1f}M  backbone:{n_bb:.1f}M  "
      f"MHA:{n_mha:.0f}K  decoder+heads:{n_dec:.1f}M")

with torch.no_grad():
    dummy = torch.zeros(2, CFG["n_input_channels"], 224, 224).to(DEVICE)
    preds = model(dummy)
    print(f"Forward pass ✓  VRAM:{torch.cuda.memory_allocated()/1e9:.2f} GB  "
          f"preds:{[tuple(p.shape) for p in preds]}")
    torch.cuda.empty_cache()


# %% ── CELL 10b: Loss functions ───────────────────────────────────────────────

def boundary_loss(logits, gt, kernel_size=5):
    """
    Boundary-weighted BCE.
    Pixels on the liver boundary receive weight 1+4=5; all others 1.

    FIX-12  Correct morphological boundary = dilation minus erosion.
    Old code:  (conv(gt)>0) & (conv(1-gt)==0)
      The second condition requires zero background pixels in the
      neighbourhood, selecting pure interior points — not the boundary.
    New code:  max_pool(gt) - (-max_pool(-gt)) > 0.5
      dilated = foreground expanded outward by kernel//2 pixels
      eroded  = foreground shrunk inward  by kernel//2 pixels
      boundary = the ring of pixels between dilated and eroded masks
    """
    gt_f     = gt.float()
    pad      = kernel_size // 2
    dilated  = F.max_pool2d(gt_f,   kernel_size, stride=1, padding=pad)
    eroded   = -F.max_pool2d(-gt_f, kernel_size, stride=1, padding=pad)
    boundary = ((dilated - eroded) > 0.5).float()
    weight   = 1.0 + 4.0 * boundary
    return F.binary_cross_entropy_with_logits(logits.float(), gt_f,
                                               weight=weight, reduction='mean')


def _weighted_iou_bce(logit, gt, pos_weight):
    """
    Weighted IoU + weighted BCE  (DS²Net paper, eq. 9-10).

    FIX-1a  avg_pool2d kernel hard-capped at 31.
      Without the cap, H=224 produced k=223, padding=111.  The pooled
      output near the padding boundary can exceed 1.0, and after
      |gt - pool(gt)| the weight map overflows bf16 → NaN BCE at step 0.

    FIX-1b  Weight map clamped to [1, 5].
      An extra safety net for edge cases where pooling with k≤31 still
      produces slight out-of-range values due to padding arithmetic.
    """
    prob = torch.sigmoid(logit.float())
    gt_f = gt.float()
    H, W = gt_f.shape[-2], gt_f.shape[-1]

    if H >= 7 and W >= 7:
        # FIX-1a: hard cap at 31, must be odd
        k   = min(31,
                  H if H % 2 == 1 else H-1,
                  W if W % 2 == 1 else W-1)
        pad = k // 2
        # FIX-1b: clamp weight to prevent bf16 overflow
        w = (1 + torch.abs(
                 gt_f - F.avg_pool2d(gt_f, k, stride=1, padding=pad)
             ).clamp(0, 1)).clamp(1, 5)
    else:
        w = torch.ones_like(gt_f)

    inter = (prob * gt_f * w).sum(dim=[2,3])
    union = ((prob + gt_f - prob*gt_f) * w).sum(dim=[2,3])
    w_iou = 1 - (inter+1)/(union+1)
    w_bce = F.binary_cross_entropy_with_logits(
        logit.float(), gt_f,
        pos_weight=torch.tensor([pos_weight], device=gt.device),
        reduction='mean')
    return w_iou.mean() + w_bce


def ds2_adaptive_loss_liver(preds, gt, pos_weight=5., boundary_weight=0.3):
    """
    DS² adaptive loss — liver only.

    Paper §III-C:
      u_i  = mean(1 − |sigmoid(p_i) − 0.5| / 0.5)   [eq.6]
      λ_i  = softmax(u) / max(softmax(u))             [eq.7-8]
      L    = Σ λ_i × (L_wIoU + L_wBCE)               [eq.9-10]

    Boundary loss added only at head0 (finest scale); coarser heads
    produce noisier boundary maps and destabilise early-training gradients.
    """
    with torch.no_grad():
        u     = torch.tensor([
                    (1-(torch.sigmoid(p.float())-0.5).abs()/0.5).mean().item()
                    for p in preds]).clamp(min=1e-6)
        ub    = torch.softmax(u, dim=0)
        lams  = (ub / ub.max().clamp(min=1e-6)).tolist()

    total = 0.
    for i, (p, lam) in enumerate(zip(preds, lams)):
        base = _weighted_iou_bce(p, gt.float(), pos_weight)
        if i == len(preds)-1:          # head0 only
            base = base + boundary_weight * boundary_loss(p, gt.float())
        total += lam * base
    return total


# %% ── CELL 11: Metrics ───────────────────────────────────────────────────────
SEG_THR = CFG["seg_threshold"]

def dice_from_probs(probs, gt, thr=None):
    thr = SEG_THR if thr is None else thr
    pb  = (probs > thr).float()
    i   = (pb * gt).sum((-2,-1))
    u   = pb.sum((-2,-1)) + gt.sum((-2,-1))
    return ((2*i+1e-6)/(u+1e-6)).mean().item()

def compute_metrics(probs, gt, thr=None):
    thr  = SEG_THR if thr is None else thr
    pred = (probs > thr).float()
    tp   = (pred*gt).sum((-1,-2))
    fp   = (pred*(1-gt)).sum((-1,-2))
    fn   = ((1-pred)*gt).sum((-1,-2))
    pr   = ((tp+1e-6)/(tp+fp+1e-6)).mean().item()
    rc   = ((tp+1e-6)/(tp+fn+1e-6)).mean().item()
    return {
        "Dice"      : ((2*tp+1e-6)/(2*tp+fp+fn+1e-6)).mean().item(),
        "IoU"       : ((tp+1e-6)/(tp+fp+fn+1e-6)).mean().item(),
        "Precision" : pr, "Recall": rc,
        "F1"        : 2*pr*rc/(pr+rc+1e-6),
        "MAE"       : (probs-gt).abs().mean().item(),
    }

def print_metrics_table(title, metrics):
    print(f"\n{'─'*58}\n  {title}\n{'─'*58}")
    print(f"  {'Class':<12}" + "".join(f"{k:>10}" for k in METRIC_KEYS))
    print("─"*58)
    for name, vals in metrics.items():
        print(f"  {name:<12}" + "".join(f"{vals[k]:>10.4f}" for k in METRIC_KEYS))
    print("─"*58)

# %% ── CELL 12: Training loop ─────────────────────────────────────────────────
use_amp   = DEVICE == "cuda"
amp_dtype = (torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported())
             else torch.float16)
scaler    = (GradScaler(init_scale=1024)
             if (use_amp and amp_dtype == torch.float16) else None)
print(f"AMP dtype: {amp_dtype}  |  GradScaler: {'ON(fp16)' if scaler else 'OFF(bf16)'}")

ckpt_path   = CKPT_DIR / "ds2net_liver_best.pth"
resume_path = CKPT_DIR / "ds2net_liver_resume.pth"
GRAD_ACCUM  = CFG["grad_accum_steps"]
VAL_EVERY   = CFG["val_every"]
_spe        = _math.ceil(len(train_loader) / GRAD_ACCUM)  # opt-steps per epoch

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=CFG["lr"], epochs=CFG["epochs"],
    steps_per_epoch=_spe, pct_start=CFG["pct_start"],
    div_factor=10., final_div_factor=1e4, anneal_strategy="cos")

start_epoch = 1; best_dice = 0.; best_metrics = None
no_improve  = 0; _gsteps   = 0

if resume_path.exists():
    print("Resuming from checkpoint ...")
    ck = torch.load(resume_path, map_location=DEVICE)
    start_epoch = ck["epoch"] + 1
    best_dice   = ck["best_dice"]
    best_metrics= ck.get("best_metrics")
    no_improve  = ck.get("no_improve", 0)
    _gsteps     = ck.get("global_opt_steps", 0)
    model.load_state_dict(ck["model"], strict=False)
    try: optimizer.load_state_dict(ck["optimizer"])
    except (ValueError, KeyError) as e: print(f"  Optimizer skipped: {e}")

    # FIX-5: restore scheduler state so LR curve is continuous.
    # Old code rebuilt from scratch with pct_start=0.10, which lost the
    # elapsed step count and produced a wrong LR immediately after resume.
    remaining = max(CFG["epochs"] - start_epoch + 1, 1)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=CFG["lr"], epochs=remaining,
        steps_per_epoch=_spe, pct_start=0.05,
        div_factor=10., final_div_factor=1e4, anneal_strategy="cos")
    if "scheduler" in ck:
        try:
            saved_sd    = ck["scheduler"]
            # FIX-15  Guard against IndexError when the saved last_epoch
            # exceeds the new scheduler's total_steps.  This happens when
            # resuming late in training (e.g. epoch 58/60) where remaining=2
            # produces a scheduler with fewer total_steps than last_epoch.
            saved_last  = saved_sd.get("last_epoch", 0)
            new_total   = scheduler.total_steps
            if saved_last <= new_total:
                scheduler.load_state_dict(saved_sd)
                print(f"  Scheduler state restored ✓  "
                      f"(last_epoch={saved_last}/{new_total})")
            else:
                print(f"  Scheduler state skipped — "
                      f"saved last_epoch={saved_last} > "
                      f"new total_steps={new_total}. "
                      f"LR will follow fresh curve from epoch 0.")
        except Exception as e:
            print(f"  Scheduler state skipped ({e}) — rebuilding")
    print(f"  epoch:{ck['epoch']}  best_dice:{best_dice:.4f}  "
          f"no_improve:{no_improve}  remaining:{remaining}  gsteps:{_gsteps}")
else:
    print(f"Fresh run — {_spe} opt-steps/epoch × {CFG['epochs']} epochs")

history = {k: [] for k in ["loss","val_loss","train_dice","val_dice",
                             "val_precision","val_recall","val_f1","lr"]}
last_vloss = last_vd = 0.
last_metrics = {m: 0. for m in METRIC_KEYS}

print(f"\nTraining DS²Net-Liver  epochs {start_epoch}→{CFG['epochs']}\n")

for epoch in range(start_epoch, CFG["epochs"]+1):
    model.train()
    ep_loss = ep_dice = 0.
    t0 = time.time()
    optimizer.zero_grad(set_to_none=True)

    for step, (imgs, masks) in enumerate(train_loader):
        imgs  = imgs.to(DEVICE,  non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        if use_amp:
            with autocast("cuda", dtype=amp_dtype):
                preds = model(imgs)
            loss = ds2_adaptive_loss_liver(
                [p.float() for p in preds], masks.float(),
                CFG["pos_weight"], CFG["boundary_weight"]) / GRAD_ACCUM
        else:
            preds = model(imgs)
            loss  = ds2_adaptive_loss_liver(preds, masks,
                        CFG["pos_weight"], CFG["boundary_weight"]) / GRAD_ACCUM

        sl = loss.item() * GRAD_ACCUM
        ok = _math.isfinite(sl) and sl < 1000.
        if not ok:
            optimizer.zero_grad(set_to_none=True)
            if not _math.isfinite(sl):
                print(f"  NaN/Inf ep{epoch} step{step} — skipping")
            else:
                print(f"  Loss spike ep{epoch} step{step} ({sl:.2e}) — skipping")
            sl = 0.
        else:
            (scaler.scale(loss).backward() if scaler else loss.backward())

        at_accum = ((step+1) % GRAD_ACCUM == 0 or (step+1) == len(train_loader))
        if ok and at_accum:
            if scaler:
                scaler.unscale_(optimizer)
                gok = all(p.grad is None or torch.isfinite(p.grad).all()
                          for p in model.parameters())
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["max_grad_norm"])
                if gok: scaler.step(optimizer); _gsteps += 1
                else:   optimizer.zero_grad(set_to_none=True)
                scaler.update()
            else:
                gok = all(p.grad is None or torch.isfinite(p.grad).all()
                          for p in model.parameters())
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["max_grad_norm"])
                if gok: optimizer.step(); _gsteps += 1
                else:   optimizer.zero_grad(set_to_none=True)
            if _gsteps >= 1: scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        ep_loss += sl
        with torch.no_grad():
            ep_dice += dice_from_probs(torch.sigmoid(sum(preds)/len(preds)), masks)

    tl  = ep_loss / len(train_loader)
    td  = ep_dice / len(train_loader)
    lr  = optimizer.param_groups[0]["lr"]

    # Reset NaN BN running stats
    nr = 0
    for m in model.modules():
        if (isinstance(m, nn.BatchNorm2d)
                and hasattr(m, 'running_mean')
                and m.running_mean is not None
                and not torch.isfinite(m.running_mean).all()):
            m.running_mean.zero_()
            m.running_var.fill_(1.)
            nr += 1
    if nr: print(f"  WARNING: reset {nr} BN modules (epoch {epoch})")

    do_val = (epoch % VAL_EVERY == 0) or epoch == CFG["epochs"]
    if do_val:
        model.eval(); vl = vd = 0.; agg = {m: 0. for m in METRIC_KEYS}; nb = 0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                if use_amp:
                    with autocast("cuda", dtype=amp_dtype): preds = model(imgs)
                else: preds = model(imgs)
                probs = torch.sigmoid(sum(preds)/len(preds))
                _vl = ds2_adaptive_loss_liver(
                    [p.float() for p in preds], masks.float(),
                    CFG["pos_weight"], CFG["boundary_weight"]).item()
                if _math.isfinite(_vl): vl += _vl
                vd += dice_from_probs(probs, masks)
                for m, v in compute_metrics(probs, masks).items(): agg[m] += v
                nb += 1
        mv = max(len(val_loader),1)
        last_vloss   = vl/mv; last_vd = vd/mv
        last_metrics = {m: agg[m]/nb for m in METRIC_KEYS}
        saved = ""
        if last_vd > best_dice:
            best_dice = last_vd; best_metrics = dict(last_metrics)
            no_improve = 0; torch.save(model.state_dict(), ckpt_path)
            saved = f"  ✓ best={best_dice:.4f}"
        else: no_improve += 1
        print(f"Ep {epoch:3d}/{CFG['epochs']} | loss {tl:.4f} | vloss {last_vloss:.4f} | "
              f"tdice {td:.4f} | vdice {last_vd:.4f} | lr {lr:.2e} | "
              f"{time.time()-t0:.0f}s{saved}")
        if no_improve >= CFG["early_stop_patience"]:
            print(f"\nEarly stopping epoch {epoch}.")
            torch.save({"epoch":epoch,"model":model.state_dict(),
                        "optimizer":optimizer.state_dict(),
                        "scheduler":scheduler.state_dict(),   # FIX-5
                        "best_dice":best_dice,"best_metrics":best_metrics,
                        "no_improve":no_improve,"global_opt_steps":_gsteps}, resume_path)
            break
    else:
        print(f"Ep {epoch:3d}/{CFG['epochs']} | loss {tl:.4f} | tdice {td:.4f} | "
              f"lr {lr:.2e} | {time.time()-t0:.0f}s  [no val]")

    history["loss"].append(tl); history["lr"].append(lr)
    history["train_dice"].append(td); history["val_loss"].append(last_vloss)
    history["val_dice"].append(last_vd)
    history["val_precision"].append(last_metrics.get("Precision",0.))
    history["val_recall"].append(last_metrics.get("Recall",0.))
    history["val_f1"].append(last_metrics.get("F1",0.))
    torch.save({"epoch":epoch,"model":model.state_dict(),
                "optimizer":optimizer.state_dict(),
                "scheduler":scheduler.state_dict(),           # FIX-5
                "best_dice":best_dice,"best_metrics":best_metrics,
                "no_improve":no_improve,"global_opt_steps":_gsteps}, resume_path)

print(f"\nBest Val Liver Dice: {best_dice:.4f}")
if best_metrics: print_metrics_table("Best Validation Metrics", {"liver": best_metrics})


# %% ── CELL 13: Training curves ───────────────────────────────────────────────
if not history["loss"]:
    print("No history — run Cell 12 first.")
else:
    ep = range(1, len(history["loss"])+1)
    fig, axes = plt.subplots(2, 2, figsize=(14,8))

    axes[0,0].plot(ep, history["loss"], label="Train loss")
    if any(v != 0 for v in history["val_loss"]):
        axes[0,0].plot(ep, history["val_loss"], label="Val loss", ls='--')
    axes[0,0].set_title("Loss"); axes[0,0].legend()

    axes[0,1].plot(ep, history["train_dice"], label="Train Dice")
    if any(v != 0 for v in history["val_dice"]):
        axes[0,1].plot(ep, history["val_dice"], label="Val Dice")
        bv = max(v for v in history["val_dice"] if v != 0)
        axes[0,1].axhline(bv, color='r', ls=':', label=f"Best:{bv:.4f}")
    axes[0,1].set_title("Liver Dice"); axes[0,1].legend()

    if any(v != 0 for v in history["val_precision"]):
        axes[1,0].plot(ep, history["val_precision"], label="Precision")
        axes[1,0].plot(ep, history["val_recall"],    label="Recall")
        axes[1,0].plot(ep, history["val_f1"],        label="F1", ls='--')
    axes[1,0].set_title("Liver P/R/F1"); axes[1,0].set_ylim(0,1); axes[1,0].legend()

    axes[1,1].plot(ep, history["lr"], color='#9C27B0')
    axes[1,1].set_title("LR (OneCycleLR)")

    plt.suptitle("DS²Net-Liver — pure DS²Net + MHA at d4")
    plt.tight_layout()
    plt.savefig(CKPT_DIR / "ds2net_liver_curves.png", dpi=150)
    plt.show()


# %% ── CELL 14: Test-set evaluation ──────────────────────────────────────────
_skip = not ckpt_path.exists()
if _skip:
    print(f"No checkpoint at {ckpt_path}. Run Cell 12 first.")
else:
    print("Loading best checkpoint ...")
    base  = model._orig_mod if hasattr(model, '_orig_mod') else model
    state = {k.replace("_orig_mod.",""):v
             for k,v in torch.load(ckpt_path, map_location=DEVICE).items()}
    base.load_state_dict(state, strict=False); base.eval(); model = base

    agg = {m: 0. for m in METRIC_KEYS}; nb = 0
    with torch.no_grad():
        for imgs, masks in tqdm(test_loader, desc="Test eval"):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            if use_amp:
                with autocast("cuda", dtype=amp_dtype): preds = model(imgs)
            else: preds = model(imgs)
            probs = torch.sigmoid(sum(preds)/len(preds))
            for m, v in compute_metrics(probs, masks).items(): agg[m] += v
            nb += 1
    print_metrics_table("TEST SET — DS²Net-Liver", {"liver": {m:agg[m]/nb for m in METRIC_KEYS}})


# %% ── CELL 15: TTA inference → save liver masks (all cases) ─────────────────
# Runs 8-fold TTA on every case (train + val + test) and saves:
#   {LIVER_DIR}/{case_id}_liver_prob.npy   float32 (H, W, D)
#   {LIVER_DIR}/{case_id}_liver_mask.npy   uint8   (H, W, D)
# Stage 2 (ds2net_v5_tumor.py) reads these as the liver spatial prior.

SEG_BATCH = 16


def _build_20ch_slices_case(pvp_path, cfg):
    """Reconstruct the 20-channel pseudo-3D stack for a single case."""
    case_dir   = Path(pvp_path).parent.parent
    pvp_img    = nib.load(str(pvp_path))
    pvp_vol    = pvp_img.get_fdata(dtype=np.float32)
    spacing    = tuple(float(z) for z in pvp_img.header.get_zooms()[:3])
    factors    = tuple(s/cfg["target_spacing"] for s in spacing)
    orig_shape = pvp_vol.shape
    n_ctx, half, sz = cfg["n_context_slices"], cfg["n_context_slices"]//2, (cfg["img_size"],)*2

    phase_vols = []
    for fname in PHASE_FILES_SEG:
        hu_min, hu_max = cfg["hu_windows"][fname]
        ph = case_dir/"NIFTI"/fname
        vol = nib.load(str(ph)).get_fdata(dtype=np.float32) if ph.exists() else pvp_vol.copy()
        if vol.shape != pvp_vol.shape:
            vol = zoom(vol, tuple(p/v for p,v in zip(pvp_vol.shape,vol.shape)), order=1)
        if any(abs(f-1.)>0.05 for f in factors): vol = zoom(vol, factors, order=1)
        vol = (np.clip(vol, hu_min, hu_max) - hu_min)/(hu_max-hu_min+1e-8)
        # Match Cell 7 FIX-10: clip to [0,1] after zoom to prevent spline overshoot
        vol = np.clip(vol, 0.0, 1.0)
        phase_vols.append(vol.astype(np.float32))

    D_rs, rs_H, rs_W = phase_vols[0].shape[2], phase_vols[0].shape[0], phase_vols[0].shape[1]
    slices = []
    for z in range(D_rs):
        ch = []
        for dz in range(-half, half+1):
            zc = max(0, min(D_rs-1, z+dz))
            for vol in phase_vols:
                ch.append(sk_resize(vol[:,:,zc], sz, order=1, preserve_range=True).astype(np.float32))
        slices.append(torch.from_numpy(np.stack(ch,0)).float())
    return torch.stack(slices), rs_H, rs_W, D_rs, orig_shape


def tta_predict_liver(model, all_slices, cfg, use_amp):
    """
    8-fold TTA: identity, flip-LR, flip-UD, rot90×3, flip+rot×2.
    Inverses verified correct:
      t6: apply = flip[-1]→rot90(+1)   invert = rot90(-1)→flip[-1]
      t7: apply = flip[-2]→rot90(+1)   invert = rot90(-1)→flip[-2]
    """
    D = all_slices.shape[0]

    def run(batch):
        b = batch.to(DEVICE)
        with torch.no_grad():
            if use_amp:
                with autocast("cuda", dtype=amp_dtype): preds = model(b)
            else: preds = model(b)
        return torch.sigmoid(sum(preds)/len(preds))

    def fwd(x, t):
        if t==0: return x
        if t==1: return torch.flip(x,[-1])
        if t==2: return torch.flip(x,[-2])
        if t==3: return torch.rot90(x, 1,[-2,-1])
        if t==4: return torch.rot90(x, 2,[-2,-1])
        if t==5: return torch.rot90(x, 3,[-2,-1])
        if t==6: return torch.rot90(torch.flip(x,[-1]), 1,[-2,-1])
        if t==7: return torch.rot90(torch.flip(x,[-2]), 1,[-2,-1])

    def inv(p, t):
        if t==0: return p
        if t==1: return torch.flip(p,[-1])
        if t==2: return torch.flip(p,[-2])
        if t==3: return torch.rot90(p,-1,[-2,-1])
        if t==4: return torch.rot90(p,-2,[-2,-1])
        if t==5: return torch.rot90(p,-3,[-2,-1])
        if t==6: return torch.flip(torch.rot90(p,-1,[-2,-1]),[-1])
        if t==7: return torch.flip(torch.rot90(p,-1,[-2,-1]),[-2])

    accum = None
    for t in range(8):
        parts = [inv(run(fwd(all_slices[s:s+SEG_BATCH], t)), t).cpu()
                 for s in range(0, D, SEG_BATCH)]
        t_all = torch.cat(parts, 0)
        accum = t_all if accum is None else accum + t_all
    return accum / 8   # (D, 1, H, W)


def connected_component_filter(mask_3d, min_voxels):
    """
    Remove 3-D components smaller than min_voxels (6-connectivity).
    FIX-8: removed unreachable n_components==0 branch (already guarded
    by the mask_3d.sum()==0 check on entry).
    """
    if min_voxels <= 0 or mask_3d.sum() == 0:
        return mask_3d
    labeled, _ = cc_label(mask_3d)         # FIX-8: discard n_components
    sizes      = np.bincount(labeled.ravel()); sizes[0] = 0
    return (sizes >= min_voxels)[labeled].astype(np.uint8)


def save_liver_mask_for_case(pvp_path, case_id, cfg):
    """
    TTA inference for one case → CC filter → save prob + mask.
    Skips if both output files already exist (idempotent / resume-safe).
    """
    prob_out = LIVER_DIR / f"{case_id}_liver_prob.npy"
    mask_out = LIVER_DIR / f"{case_id}_liver_mask.npy"
    if prob_out.exists() and mask_out.exists():
        return np.load(str(mask_out))

    all_slices, rs_H, rs_W, D_rs, orig_shape = _build_20ch_slices_case(pvp_path, cfg)
    model.eval()
    avg_p = tta_predict_liver(model, all_slices, cfg, use_amp)

    # Resize from img_size → resampled spatial resolution
    prob_rs = np.zeros((rs_H, rs_W, D_rs), dtype=np.float32)
    for d in range(D_rs):
        prob_rs[:,:,d] = sk_resize(avg_p[d,0].numpy(), (rs_H,rs_W),
                                   order=1, preserve_range=True).astype(np.float32)
    # Zoom back to original voxel space
    prob_vol = (zoom(prob_rs, tuple(o/r for o,r in zip(orig_shape,prob_rs.shape)), order=1)
                .astype(np.float32) if prob_rs.shape != orig_shape else prob_rs)

    mask = connected_component_filter(
        (prob_vol > cfg["seg_threshold"]).astype(np.uint8),
        cfg.get("min_component_voxels",{}).get("default",1000))

    np.save(str(prob_out), prob_vol)
    np.save(str(mask_out), mask)
    return mask


if _skip:
    print("Skipping TTA — run Cell 12 first.")
else:
    print(f"\nRunning TTA liver inference on ALL {len(df)} cases ...")
    print(f"Output: {LIVER_DIR}\n")
    failed = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Liver TTA"):
        t_case = time.time()
        try:
            save_liver_mask_for_case(row["pvp_path"], row["case_id"], CFG)
            tqdm.write(f"  {row['case_id']} ({row['tumor_type']}) ✓  "
                       f"{time.time()-t_case:.1f}s")
        except Exception as e:
            tqdm.write(f"  FAILED {row['case_id']} — {type(e).__name__}: {e}")
            failed.append(row["case_id"])

    saved = len(list(LIVER_DIR.glob("*_liver_mask.npy")))
    print(f"\nDone.  {saved}/{len(df)} liver masks saved → {LIVER_DIR}")
    if failed: print(f"Failed ({len(failed)}): {failed}")
    print("\n→ Run ds2net_v5_tumor.py (Stage 2) next.")
